In [4]:
import json
import pandas as pd
import re
from collections import defaultdict
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

import os
def get_indiv_concepts(formula) -> list:
    concepts = []
    concps = re.findall(r'(?<!\bNOT\s)(?:\b(?:hyp|pre|oth):[^\s)]+)', formula)
    for c in concps:
        try:
            end_idx = c.index(')')
        except:
            end_idx = len(c)
        concepts.append(c[:end_idx])
    return concepts

# 1. Open JSON and look only at cluster3
with open('/workspace/CCE_NLI/BERT/formula_masks/lottery_ticket/Run0.25_5/formula_masks_0.0.json', 'r') as f:
    data = json.load(f)

cluster3 = data['3']

active_neurons = []
for neuron, values in cluster3.items():
    if values[3] == 1:
        active_neurons.append(int(neuron))

print(f"Neurons in cluster3 with index 3 == 1: {active_neurons}")

# 2. Open CSV and get bestname for each active neuron
df = pd.read_csv('/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5_3clusters/Expls/0.0%Pruned/Cluster3IOUS1024N.csv')

all_concepts = []
for neuron in active_neurons:
    row = df[df['unit'] == neuron]
    if not row.empty:
        bestname = row['best_name'].values[0]

        # 3. Call get_indiv_concepts and extend the list
        concepts = get_indiv_concepts(bestname)
        all_concepts.extend(concepts)
    else:
        print(f"Neuron {neuron} not found in CSV")

print(f"Total concepts collected: {len(all_concepts)}")
print(all_concepts)

Neurons in cluster3 with index 3 == 1: [33, 37, 39, 42, 45, 49, 53, 64, 74, 89, 109, 129, 132, 136, 153, 183, 206, 216, 218, 220, 232, 259, 269, 278, 300, 313, 314, 317, 349, 380, 389, 416, 458, 476, 483, 485, 487, 491, 502, 504, 511, 538, 540, 547, 548, 575, 593, 623, 631, 637, 655, 664, 678, 690, 710, 740, 743, 789, 809, 832, 846, 861, 865, 873, 891, 901, 905, 921, 922, 936, 950, 961, 963, 956, 978, 995, 1008]
Total concepts collected: 286
['hyp:tag:prp$', 'hyp:tok:for', 'hyp:tag:vb', 'hyp:tok:playing', 'pre:tok:air', 'hyp:tok:sleeping', 'pre:tok:standing', 'hyp:tok:men', 'pre:tok:waiting', 'hyp:tok:lake', 'hyp:tok:for', 'pre:tok:blue', 'hyp:tok:tall', 'hyp:tok:to', 'oth:overlap:overlap75', 'pre:tok:blue', 'hyp:tok:park', 'hyp:tok:playing', 'hyp:tok:in', 'hyp:tok:eating', 'hyp:tok:inside', 'hyp:tok:watching', 'hyp:tok:for', 'hyp:tok:her', 'hyp:tok:tall', 'hyp:tok:sad', 'hyp:tok:their', 'hyp:tag:prp$', 'hyp:tok:to', 'hyp:tok:for', 'hyp:tok:woman', 'pre:tok:shirt', 'pre:tok:blue', 'hyp

In [55]:
import json
import pandas as pd
import re
from collections import defaultdict
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import os
from itertools import combinations

def get_indiv_concepts(formula) -> list:
    concepts = []
    concps = re.findall(r'(?<!\bNOT\s)(?:\b(?:hyp|pre|oth):[^\s)]+)', formula)
    for c in concps:
        try:
            end_idx = c.index(')')
        except:
            end_idx = len(c)
        concepts.append(c[:end_idx])
    return concepts

def get_groups(formula):
    indiv = get_indiv_concepts(formula)
    groups = []
    for length in range(2, len(indiv) + 1):
        for combo in combinations(indiv, length):
            groups.append(tuple(sorted(combo)))
    return sorted(groups)

# 1. Open JSONs
with open('/workspace/CCE_NLI/BERT/formula_masks/CoFi/Run0.25_5/formula_masks_0.0.json', 'r') as f:
    data_00 = json.load(f)
with open('/workspace/CCE_NLI/BERT/formula_masks/CoFi/Run0.25_5/formula_masks_0.25267370011269075.json', 'r') as f:
    data_025 = json.load(f)

cluster_00  = data_00['3']
cluster_025 = data_025['3']

# 2. Open CSVs
df_00  = pd.read_csv('/workspace/CCE_NLI/BERT/exp/CoFi/Run0.25_5/Expls/0.0%Pruned/Cluster3IOUS1024N.csv')
df_025 = pd.read_csv('/workspace/CCE_NLI/BERT/exp/CoFi/Run0.25_5/Expls/0.25267370011269075%Pruned/Cluster3IOUS1024N.csv')

# 3. Loop over all 10000 samples
NUM_SAMPLES = 10
total_ratio = 0.0
valid_samples = 0
total_iou=0.0
total_carried_over=0.0
for sample_idx in range(NUM_SAMPLES):

    # Get active neurons for this sample
    active_00 = []
    for neuron, values in cluster_00.items():
        if values[sample_idx] == 1:
            active_00.append(int(neuron))

    active_025 = []
    for neuron, values in cluster_025.items():
        if values[sample_idx] == 1:
            active_025.append(int(neuron))

    # Get concepts for 0.0
    concepts_00 = []
    for neuron in active_00:
        row = df_00[df_00['unit'] == neuron]
        if not row.empty:
            bestname = row['best_name'].values[0]
            concepts = get_indiv_concepts(bestname)
            concepts_00.extend(concepts)

    # Get concepts for 0.25
    concepts_025 = []
    for neuron in active_025:
        row = df_025[df_025['unit'] == neuron]
        if not row.empty:
            bestname = row['best_name'].values[0]
            concepts = get_indiv_concepts(bestname)
            concepts_025.extend(concepts)

    # Compute ratio len(00) & len(025) / len(025)
    if len(concepts_025) > 0:
        iou = (len(set(concepts_00) & set(concepts_025))) / len(set(concepts_025) |set(concepts_00)) 
        ratio = (len(set(concepts_00) & set(concepts_025))) / len(set(concepts_025)) 
        carried_over = (len(set(concepts_00) & set(concepts_025))) / len(set(concepts_00)) 
        total_ratio += ratio
        total_iou += iou
        total_carried_over += carried_over
        valid_samples += 1

# 4. Average ratio across all valid samples
avg_ratio = total_ratio / valid_samples if valid_samples > 0 else 0
avg_iou = total_iou / valid_samples if valid_samples > 0 else 0
avg_carried_over = carried_over / valid_samples if valid_samples > 0 else 0
print(f"Valid samples: {valid_samples}")
print(f"Average % of concepts in 25 that was in 0 between dense and sparse concepts across all samples (per sample): {avg_ratio}")
print(f"Average % iou between dense and sparse concepts across all samples (per sample): {avg_iou}")
print(f"Average % of concepts that was carried over from 0 to 25 between dense and sparse concepts across all samples (per sample): {avg_carried_over}")

NameError: name 'total_iou' is not defined

In [48]:
cofi_bert_25_sample3=['oth:overlap:overlap50', 'pre:tok:blue', 'hyp:tok:holding', 'hyp:tag:in', 'pre:tok:men', 'hyp:tok:eating', 'pre:tok:standing', 'pre:tok:walking', 'hyp:tok:alone', 'hyp:tok:swimming', 'hyp:tok:wearing', 'pre:tok:blue', 'hyp:tok:book', 'pre:tok:cowboy', 'oth:overlap:overlap75', 'pre:tok:blue', 'hyp:tok:outside', 'hyp:tok:performing', 'hyp:tok:her', 'pre:tok:black', 'hyp:tok:tall', 'pre:tok:football', 'hyp:tok:their', 'pre:tag:dt', 'hyp:tok:something', 'oth:overlap:overlap75', 'pre:tok:blue', 'hyp:tok:outside', 'pre:tok:microphone', 'hyp:tag:ex', 'hyp:tok:outside', 'pre:tok:blue', 'pre:tok:running', 'pre:tok:cooking', 'hyp:tok:day', 'hyp:tag:in', 'hyp:tok:sleeping', 'hyp:tok:nobody', 'hyp:tok:cat', 'hyp:tok:outside', 'pre:tok:black', 'pre:tok:blue', 'hyp:tok:running', 'pre:tag:jj', 'hyp:tok:outside', 'hyp:tok:outdoors', 'hyp:tag:prp$', 'hyp:tag:vb', 'pre:tok:football', 'hyp:tok:pool', 'pre:tag:jj', 'pre:tok:black', 'hyp:tok:outside', 'hyp:tok:in', 'pre:tok:sitting', 'pre:tok:mountain', 'oth:overlap:overlap50', 'pre:tok:blue', 'pre:tag:jj', 'hyp:tok:outside', 'hyp:tok:outdoors', 'pre:tok:blue', 'pre:tok:water', 'pre:tok:cooking', 'pre:tok:singing', 'pre:tok:bench', 'hyp:tag:nnp', 'hyp:tag:prp$', 'hyp:tok:after', 'hyp:tok:competition', 'pre:tok:dancing', 'hyp:tag:prp$', 'hyp:tok:boy', 'hyp:tok:crowd', 'pre:tok:orange', 'hyp:tok:dancing', 'hyp:tok:in', 'hyp:tok:sleeping', 'hyp:tok:watching', 'hyp:tok:driving', 'hyp:tag:prp$', 'pre:tok:beach', 'hyp:tok:tall', 'hyp:tok:park', 'hyp:tok:fishing', 'pre:tok:blue', 'pre:tok:girl', 'hyp:tok:home', 'pre:tok:drinking', 'pre:tok:children', 'pre:tok:girl', 'hyp:tok:child', 'hyp:tok:snow', 'hyp:tok:park', 'hyp:tag:prp$', 'pre:tok:baseball', 'pre:tok:sand', 'hyp:tok:making', 'hyp:tok:tall', 'hyp:tag:.', 'pre:tok:young', 'pre:tok:children', 'hyp:tag:prp$', 'hyp:tok:for', 'hyp:tok:party', 'pre:tok:headphones', 'pre:tok:in', 'hyp:tok:eating', 'hyp:tag:prp$', 'oth:overlap:overlap50', 'hyp:tag:prp$', 'hyp:tok:in', 'hyp:tok:dancing', 'pre:tok:girls', 'oth:overlap:overlap50', 'hyp:tok:sitting', 'hyp:tok:kids', 'hyp:tag:vb', 'hyp:tok:for', 'hyp:tok:their', 'hyp:tok:day', 'hyp:tok:birthday', 'pre:tok:standing', 'pre:tok:walking', 'pre:tok:hat', 'pre:tok:walks', 'pre:tok:playing', 'pre:tok:red', 'pre:tok:children', 'pre:tok:plays', 'pre:tag:``', 'pre:tok:blue', 'hyp:tok:tall', 'hyp:tok:sleeping', 'pre:tok:blue', 'pre:tok:pink', 'pre:tok:shorts', 'hyp:tok:shirt', 'hyp:tok:nobody', 'hyp:tag:prp$', 'hyp:tok:riding', 'hyp:tok:party', 'hyp:tag:nnp', 'hyp:tok:competition', 'hyp:tok:in', 'hyp:tok:running', 'hyp:tok:sleeping', 'hyp:tok:swimming', 'hyp:tok:in', 'pre:tok:bed', 'hyp:tok:running', 'pre:tok:in', 'hyp:tok:sitting', 'hyp:tag:prp$', 'hyp:tok:running', 'pre:tok:children', 'pre:tok:football', 'hyp:tok:playing', 'oth:overlap:overlap50', 'hyp:tok:outside', 'hyp:tok:kids', 'hyp:tok:in', 'hyp:tok:inside', 'hyp:tok:sleeping', 'hyp:tok:eating', 'hyp:tok:outside', 'pre:tok:blue', 'hyp:tag:ex', 'hyp:tok:outdoors', 'hyp:tok:indoors', 'hyp:tag:prp$', 'hyp:tok:for', 'hyp:tok:park', 'hyp:tok:to', 'hyp:tok:her', 'hyp:tok:their', 'hyp:tok:concert', 'hyp:tok:party', 'hyp:tok:dancing', 'hyp:tok:boy', 'hyp:tok:his', 'pre:tok:children', 'hyp:tok:tall', 'pre:tok:boys', 'pre:tok:black', 'pre:tok:blue', 'hyp:tok:for', 'hyp:tok:trying', 'hyp:tok:outside', 'pre:tok:blue', 'hyp:tok:train', 'pre:tok:blond', 'pre:tok:green', 'hyp:tag:prp$', 'pre:tok:pool', 'hyp:tok:park', 'pre:tok:baby', 'hyp:tag:in', 'hyp:tok:dancing', 'hyp:tok:running', 'hyp:tok:riding', 'pre:tok:blue', 'hyp:tok:inside', 'pre:tok:bike', 'hyp:tag:.', 'hyp:tok:watching', 'pre:tag:nn', 'hyp:tok:there', 'pre:tok:baseball', 'hyp:tok:outdoors', 'hyp:tok:walking', 'pre:tok:children', 'hyp:tok:running', 'pre:tok:running', 'pre:tok:kids', 'hyp:tok:beach', 'pre:tok:blue', 'hyp:tok:eating', 'hyp:tok:pool', 'hyp:tok:nobody', 'oth:overlap:overlap75', 'pre:tok:standing', 'hyp:tag:ex', 'hyp:tok:water', 'hyp:tok:shirt', 'pre:tok:blue', 'pre:tok:riding', 'hyp:tok:tall', 'pre:tok:race', 'pre:tok:motorcycle', 'pre:tok:blue', 'pre:tok:sitting', 'pre:tok:smiling', 'hyp:tok:party', 'hyp:tag:in', 'hyp:tok:to', 'pre:tag:jj', 'hyp:tok:outside', 'pre:tok:soccer', 'hyp:tok:there', 'oth:overlap:overlap50', 'pre:tok:blue', 'pre:tok:on', 'pre:tok:boy', 'hyp:tok:tall', 'hyp:tag:prp$', 'hyp:tok:tall', 'hyp:tok:for', 'hyp:tok:sad', 'hyp:tok:in', 'hyp:tok:sleeping', 'hyp:tok:cat', 'pre:tok:horse', 'hyp:tag:prp$', 'hyp:tok:for', 'hyp:tag:vb', 'hyp:tok:park', 'pre:tok:blue', 'pre:tok:brown', 'hyp:tok:black', 'pre:tok:orange', 'hyp:tok:sleeping', 'pre:tok:blue', 'pre:tok:pink', 'pre:tok:standing', 'pre:tok:blue', 'pre:tok:field', 'pre:tok:snow', 'hyp:tok:car', 'hyp:tok:walking', 'pre:tok:in', 'hyp:tok:outside', 'hyp:tok:at', 'hyp:tok:for', 'hyp:tok:to', 'hyp:tok:their', 'hyp:tok:friends', 'hyp:tok:snowboarder']
cofi_bert_0_sample3=['pre:tag:jj', 'hyp:tok:swimming', 'hyp:tok:dancing', 'oth:overlap:overlap75', 'pre:tok:blue', 'hyp:tok:holding', 'pre:tok:microphone', 'oth:overlap:overlap75', 'pre:tok:race', 'pre:tok:blue', 'hyp:tok:running', 'pre:tag:in', 'pre:tok:air', 'pre:tok:standing', 'pre:tok:walking', 'hyp:tok:alone', 'hyp:tok:swimming', 'hyp:tok:wearing', 'pre:tok:blue', 'hyp:tok:holding', 'pre:tok:cowboy', 'hyp:tag:prp$', 'hyp:tok:for', 'hyp:tok:desert', 'pre:tok:girl', 'pre:tok:on', 'hyp:tok:football', 'hyp:tok:home', 'oth:overlap:overlap75', 'pre:tok:blue', 'hyp:tok:outside', 'hyp:tok:outdoors', 'pre:tok:stage', 'hyp:tok:in', 'pre:tok:dog', 'hyp:tok:football', 'pre:tok:riding', 'hyp:tag:nns', 'hyp:tag:ex', 'hyp:tok:human', 'hyp:tok:there', 'oth:overlap:overlap75', 'pre:tok:blue', 'pre:tok:green', 'hyp:tag:prp$', 'hyp:tok:boy', 'hyp:tok:park', 'hyp:tag:,', 'hyp:tok:at', 'oth:overlap:overlap50', 'pre:tok:blue', 'hyp:tok:outside', 'pre:tok:blue', 'pre:tok:white', 'hyp:tok:outside', 'pre:tok:cooking', 'hyp:tag:in', 'hyp:tok:sleeping', 'hyp:tok:cat', 'hyp:tok:nobody', 'hyp:tok:outside', 'pre:tok:black', 'pre:tok:blue', 'hyp:tok:in', 'hyp:tok:cat', 'hyp:tok:girls', 'hyp:tok:sleeping', 'pre:tag:jj', 'hyp:tok:wearing', 'hyp:tok:there', 'pre:tok:beach', 'pre:tok:blue', 'hyp:tok:walking', 'hyp:tok:sleeping', 'pre:tok:dog', 'pre:tok:girl', 'pre:tok:girls', 'pre:tok:children', 'pre:tok:smiling', 'hyp:tag:prp$', 'hyp:tok:park', 'pre:tok:football', 'hyp:tok:pool', 'hyp:tok:concert', 'hyp:tok:in', 'pre:tok:sitting', 'hyp:tok:eating', 'oth:overlap:overlap75', 'pre:tok:blue', 'pre:tag:jj', 'hyp:tok:outdoors', 'hyp:tok:outside', 'pre:tag:nns', 'pre:tok:park', 'pre:tok:person', 'pre:tag:vbg', 'hyp:tag:ex', 'pre:tok:sleeping', 'hyp:tok:dinner', 'hyp:tok:in', 'hyp:tok:eating', 'hyp:tok:inside', 'pre:tag:in', 'hyp:tok:there', 'hyp:tok:wearing', 'hyp:tag:in', 'hyp:tok:eating', 'hyp:tok:driving', 'hyp:tok:his', 'hyp:tok:to', 'hyp:tok:tall', 'hyp:tok:old', 'hyp:tok:their', 'hyp:tag:.', 'hyp:tok:sleeping', 'hyp:tok:beach', 'pre:tok:in', 'pre:tok:water', 'pre:tok:cooking', 'pre:tok:top', 'hyp:tok:at', 'pre:tok:blue', 'hyp:tok:eating', 'hyp:tok:black', 'hyp:tag:prp$', 'hyp:tok:to', 'hyp:tag:nnp', 'pre:tok:dancing', 'hyp:tok:competition', 'hyp:tag:prp$', 'hyp:tok:boy', 'pre:tok:orange', 'hyp:tok:crowd', 'hyp:tok:dancing', 'pre:tag:dt', 'hyp:tok:running', 'hyp:tok:at', 'hyp:tok:in', 'hyp:tok:driving', 'hyp:tok:couch', 'hyp:tag:prp$', 'hyp:tok:at', 'hyp:tok:for', 'pre:tok:car', 'hyp:tok:tennis', 'hyp:tok:home', 'pre:tok:blue', 'pre:tok:black', 'pre:tok:drinking', 'pre:tok:on', 'pre:tok:train', 'pre:tok:race', 'hyp:tok:and', 'hyp:tag:.', 'pre:tok:boys', 'pre:tok:boy', 'pre:tok:standing', 'pre:tok:walking', 'hyp:tok:eating', 'hyp:tok:alone', 'pre:tok:blue', 'pre:tok:stage', 'pre:tok:singing', 'hyp:tok:eating', 'pre:tok:circle', 'hyp:tag:prp$', 'hyp:tok:walking', 'pre:tok:gathered', 'pre:tok:wall', 'hyp:tok:bar', 'hyp:tag:in', 'hyp:tok:nobody', 'hyp:tok:sitting', 'pre:tag:jj', 'pre:tok:standing', 'hyp:tok:there', 'pre:tag:vbp', 'pre:tok:guitar', 'pre:tok:skateboard', 'pre:tok:football', 'pre:tag:vbg', 'pre:tok:children', 'hyp:tok:there', 'hyp:tag:prp$', 'hyp:tok:to', 'hyp:tok:party', 'hyp:tok:baseball', 'pre:tag:nn', 'pre:tok:baseball', 'pre:tok:in', 'hyp:tok:eating', 'hyp:tag:prp$', 'oth:overlap:overlap75', 'hyp:tag:nnp', 'hyp:tag:in', 'pre:tok:beach', 'pre:tok:grass', 'hyp:tok:water', 'pre:tag:vbp', 'hyp:tok:tall', 'pre:tok:pool', 'pre:tag:dt', 'pre:tok:table', 'pre:tag:nn', 'hyp:tok:black', 'hyp:tok:in', 'pre:tok:girls', 'hyp:tok:cat', 'hyp:tag:in', 'pre:tok:dog', 'hyp:tok:swimming', 'hyp:tok:eating', 'hyp:tok:in', 'hyp:tok:sleeping', 'hyp:tok:eating', 'hyp:tok:inside', 'hyp:tok:for', 'hyp:tok:to', 'pre:tok:green', 'pre:tok:hands', 'hyp:tag:vb', 'hyp:tok:for', 'hyp:tok:their', 'hyp:tok:birthday', 'hyp:tok:having', 'hyp:tag:prp$', 'oth:overlap:overlap75', 'hyp:tag:in', 'pre:tok:sand', 'hyp:tok:woman', 'pre:tok:green', 'pre:tok:standing', 'pre:tok:walking', 'pre:tok:running', 'pre:tok:standing', 'pre:tok:field', 'pre:tok:skateboard', 'hyp:tag:prp$', 'hyp:tok:for', 'hyp:tok:park', 'hyp:tok:outside', 'hyp:tok:race', 'hyp:tag:in', 'hyp:tok:eating', 'hyp:tok:swimming', 'hyp:tok:eating', 'hyp:tok:in', 'hyp:tok:sleeping', 'hyp:tok:home', 'hyp:tag:in', 'hyp:tok:dancing', 'pre:tok:desk', 'hyp:tok:birthday', 'hyp:tag:in', 'pre:tok:sitting', 'hyp:tok:swimming', 'hyp:tok:for', 'hyp:tok:to', 'pre:tok:standing', 'hyp:tok:front', 'hyp:tag:.', 'pre:tok:walking', 'pre:tok:walks', 'pre:tok:skateboarder', 'pre:tok:blue', 'pre:tok:young', 'hyp:tok:sleeping', 'hyp:tok:cat', 'pre:tok:black', 'pre:tok:blue', 'pre:tok:pink', 'hyp:tok:girls', 'hyp:tok:nobody', 'oth:overlap:overlap75', 'pre:tok:blue', 'pre:tok:baseball', 'pre:tok:jumping', 'hyp:tag:prp$', 'hyp:tok:to', 'hyp:tok:for', 'pre:tok:riding', 'pre:tok:dancing', 'hyp:tag:prp$', 'hyp:tok:for', 'hyp:tok:at', 'hyp:tok:tall', 'pre:tag:dt', 'pre:tok:park', 'pre:tok:women', 'hyp:tag:prp$', 'pre:tok:boy', 'hyp:tok:at', 'hyp:tok:party', 'pre:tok:in', 'pre:tok:running', 'hyp:tok:sitting', 'pre:tok:black', 'pre:tok:brown', 'hyp:tok:park', 'hyp:tok:soccer', 'pre:tok:wooden', 'hyp:tag:jj', 'pre:tok:children', 'hyp:tok:running', 'hyp:tok:in', 'hyp:tok:inside', 'hyp:tok:cat', 'hyp:tok:sleeping', 'pre:tag:dt', 'hyp:tok:sleeping', 'hyp:tok:to', 'pre:tok:sitting', 'hyp:tok:their', 'pre:tok:sits', 'hyp:tok:outside', 'pre:tok:blue', 'hyp:tag:ex', 'hyp:tok:outdoors', 'pre:tok:room', 'hyp:tag:prp$', 'pre:tok:performing', 'hyp:tok:party', 'pre:tok:room', 'hyp:tok:his', 'pre:tok:children', 'hyp:tok:boy', 'hyp:tag:nnp', 'pre:tok:black', 'pre:tok:blue', 'hyp:tok:for', 'pre:tok:boy', 'hyp:tok:outside', 'pre:tok:blue', 'hyp:tok:train', 'pre:tok:blond', 'pre:tok:green', 'hyp:tok:in', 'pre:tok:sitting', 'pre:tok:holding', 'pre:tag:jj', 'hyp:tok:white', 'pre:tag:vbg', 'pre:tok:sitting', 'pre:tok:sits', 'hyp:tok:outdoors', 'hyp:tag:prp$', 'pre:tok:pool', 'hyp:tok:park', 'hyp:tok:baby', 'hyp:tok:lunch', 'hyp:tok:in', 'pre:tok:boy', 'hyp:tok:sleeping', 'hyp:tag:in', 'hyp:tok:dancing', 'pre:tok:store', 'hyp:tok:in', 'pre:tok:black', 'pre:tok:horse', 'hyp:tag:prp$', 'pre:tok:white', 'pre:tag:jj', 'pre:tok:yellow', 'hyp:tag:nns', 'pre:tok:red', 'hyp:tok:for', 'pre:tag:nn', 'hyp:tok:there', 'hyp:tok:outside', 'pre:tok:baseball', 'pre:tok:children', 'pre:tok:people', 'hyp:tok:running', 'hyp:tok:street', 'pre:tok:kids', 'pre:tag:nns', 'pre:tok:pole', 'hyp:tok:kids', 'oth:overlap:overlap75', 'pre:tok:snow', 'pre:tok:book', 'hyp:tok:race', 'hyp:tok:at', 'pre:tok:black', 'pre:tok:children', 'pre:tok:cooking', 'oth:overlap:overlap75', 'pre:tok:standing', 'hyp:tok:outside', 'hyp:tok:wearing', 'hyp:tok:near', 'pre:tok:little', 'pre:tok:young', 'pre:tok:children', 'pre:tok:blue', 'pre:tok:sitting', 'pre:tok:smiling', 'hyp:tok:party', 'pre:tag:jj', 'hyp:tok:outside', 'hyp:tok:for', 'oth:overlap:overlap50', 'pre:tok:blue', 'hyp:tag:prp$', 'hyp:tag:vb', 'hyp:tok:for', 'pre:tok:mountain', 'pre:tok:rock', 'hyp:tok:baseball', 'hyp:tok:in', 'hyp:tok:home', 'pre:tok:on', 'pre:tok:young', 'hyp:tok:tall', 'pre:tag:nns', 'pre:tok:food', 'hyp:tok:wearing', 'hyp:tok:there', 'pre:tag:.', 'hyp:tag:in', 'hyp:tag:.', 'hyp:tok:park', 'pre:tok:swimming', 'hyp:tok:watching', 'pre:tok:men', 'hyp:tok:football', 'pre:tok:snow', 'pre:tok:children', 'hyp:tag:prp$', 'hyp:tok:tall', 'hyp:tok:for', 'hyp:tok:after', 'hyp:tok:in', 'hyp:tok:inside', 'hyp:tok:cat', 'hyp:tok:eating', 'hyp:tok:in', 'hyp:tok:inside', 'hyp:tok:sleeping', 'hyp:tok:nobody', 'hyp:tok:in', 'hyp:tok:watching', 'hyp:tok:horse', 'hyp:tok:sleeping', 'hyp:tag:prp$', 'hyp:tok:for', 'hyp:tok:park', 'pre:tok:headphones', 'hyp:tok:catch', 'hyp:tok:for', 'hyp:tok:to', 'hyp:tok:their', 'hyp:tok:soccer', 'hyp:tok:birthday', 'hyp:tok:kids', 'hyp:tok:outside', 'hyp:tok:standing', 'hyp:tok:park', 'pre:tok:paint', 'pre:tok:blue', 'pre:tok:brown', 'pre:tok:orange', 'pre:tok:washing', 'pre:tag:jj', 'pre:tok:and', 'hyp:tok:sleeping', 'hyp:tok:swimming', 'hyp:tok:bike', 'hyp:tok:dog', 'pre:tok:standing', 'hyp:tok:park', 'hyp:tag:in', 'hyp:tok:running', 'hyp:tok:eating', 'pre:tok:blue', 'pre:tok:white', 'pre:tok:standing', 'hyp:tok:sleeping', 'pre:tok:blue', 'pre:tok:field', 'hyp:tok:car', 'pre:tok:snow', 'pre:tok:horse', 'hyp:tag:in', 'hyp:tok:outside', 'pre:tok:dancing', 'hyp:tok:beach', 'pre:tok:young', 'hyp:tok:park', 'hyp:tok:game', 'pre:tok:water', 'pre:tag:jj', 'hyp:tok:for', 'hyp:tok:to', 'hyp:tok:their', 'hyp:tok:bike', 'hyp:tok:snowboarder']

cofi_bert_25_sample3=sorted(cofi_bert_25_sample3)
cofi_bert_0_sample3=sorted(cofi_bert_0_sample3)
print(len(set(cofi_bert_0_sample3) & set(cofi_bert_25_sample3) )/len(set(cofi_bert_25_sample3)))

#in cofi 78% of concepts hollistically are the same tho the neuron behavior is different nad highest contrib is different

cofi_bert_25_sample10=['hyp:tok:dog', 'hyp:tok:outside', 'hyp:tok:outdoors', 'hyp:tok:grass', 'hyp:tok:field', 'hyp:tok:her', 'pre:tok:black', 'hyp:tok:tall', 'pre:tok:football', 'hyp:tok:their', 'hyp:tok:there', 'oth:overlap:overlap75', 'hyp:tok:outside', 'pre:tok:football', 'hyp:tok:outdoors', 'hyp:tag:in', 'hyp:tok:sleeping', 'hyp:tok:nobody', 'hyp:tok:cat', 'pre:tag:jj', 'hyp:tok:outside', 'hyp:tok:outdoors', 'hyp:tag:prp$', 'hyp:tag:vb', 'pre:tok:football', 'hyp:tok:pool', 'pre:tag:jj', 'pre:tok:black', 'hyp:tok:outside', 'hyp:tag:nn', 'hyp:tok:driving', 'hyp:tok:at', 'hyp:tok:outside', 'hyp:tok:soccer', 'hyp:tok:dog', 'hyp:tok:dog', 'hyp:tok:outside', 'hyp:tok:outdoors', 'pre:tok:dogs', 'hyp:tok:concert', 'hyp:tag:ex', 'hyp:tok:outside', 'pre:tok:horse', 'pre:tok:football', 'hyp:tok:instrument', 'hyp:tag:nn', 'hyp:tok:sitting', 'hyp:tag:.', 'pre:tok:young', 'pre:tok:children', 'hyp:tok:eating', 'hyp:tok:swimming', 'hyp:tok:sleeping', 'hyp:tok:alone', 'hyp:tok:eats', 'hyp:tok:dog', 'hyp:tok:to', 'hyp:tok:waiting', 'hyp:tok:dog', 'pre:tok:snow', 'pre:tok:dog', 'hyp:tok:sleeping', 'pre:tok:mountain', 'hyp:tag:nn', 'hyp:tag:vbz', 'hyp:tok:man', 'pre:tok:boy', 'pre:tok:boys', 'hyp:tok:shopping', 'hyp:tok:store', 'pre:tag:jj', 'hyp:tok:outside', 'hyp:tok:walking', 'oth:overlap:overlap75', 'pre:tok:dog', 'pre:tok:football', 'hyp:tok:outside', 'pre:tok:field', 'pre:tag:jj', 'hyp:tok:outside', 'hyp:tok:tall', 'pre:tok:singing', 'pre:tok:laying', 'pre:tok:guitar', 'pre:tok:football', 'oth:overlap:overlap50', 'pre:tok:track', 'pre:tok:football', 'pre:tag:jj', 'hyp:tok:outside', 'pre:tok:man', 'pre:tok:soccer', 'pre:tok:football', 'hyp:tok:tall', 'hyp:tok:outdoors', 'hyp:tok:outside', 'hyp:tok:something', 'pre:tok:football', 'hyp:tok:wet', 'hyp:tag:prp$', 'hyp:tok:running', 'pre:tok:children', 'pre:tok:football', 'hyp:tag:ex', 'oth:overlap:overlap50', 'pre:tok:young', 'hyp:tok:there', 'pre:tok:woman', 'pre:tok:young', 'pre:tok:green', 'hyp:tok:men', 'pre:tok:boy', 'pre:tok:boys', 'pre:tok:water', 'hyp:tok:boy', 'hyp:tok:his', 'pre:tok:children', 'hyp:tok:tall', 'pre:tok:boys', 'oth:overlap:overlap50', 'pre:tok:football', 'hyp:tok:sitting', 'hyp:tok:soccer', 'pre:tok:dog', 'pre:tok:football', 'pre:tok:running', 'pre:tok:horse', 'pre:tok:game', 'hyp:tag:in', 'hyp:tok:dancing', 'hyp:tok:running', 'hyp:tok:eating', 'pre:tok:sitting', 'hyp:tok:sleeping', 'hyp:tok:eats', 'hyp:tok:alone', 'oth:overlap:overlap75', 'pre:tok:black', 'pre:tok:football', 'hyp:tok:outside', 'hyp:tok:there', 'hyp:tok:outdoors', 'pre:tok:football', 'hyp:tok:field', 'pre:tok:running', 'pre:tok:street', 'pre:tok:cart', 'pre:tok:football', 'oth:overlap:overlap50', 'pre:tok:football', 'pre:tok:game', 'hyp:tag:in', 'pre:tok:stage', 'pre:tok:air', 'hyp:tok:bike', 'hyp:tok:sleeping', 'pre:tok:running', 'hyp:tok:riding', 'pre:tok:football', 'hyp:tag:in', 'hyp:tok:to', 'pre:tag:jj', 'hyp:tok:outside', 'pre:tok:soccer', 'pre:tok:football', 'pre:tok:men', 'hyp:tok:sitting', 'pre:tok:horse', 'hyp:tok:eating', 'pre:tok:beach', 'hyp:tok:cat', 'hyp:tok:sleeping', 'hyp:tok:eats', 'pre:tok:cooking', 'pre:tok:dog', 'hyp:tok:dog', 'pre:tok:dogs', 'hyp:tok:swimming']
cofi_bert_0_sample10=['pre:tag:jj', 'hyp:tok:swimming', 'hyp:tok:dancing', 'oth:overlap:overlap75', 'pre:tok:blue', 'hyp:tok:holding', 'pre:tok:microphone', 'oth:overlap:overlap75', 'pre:tok:race', 'pre:tok:blue', 'hyp:tok:running', 'pre:tag:in', 'pre:tok:air', 'pre:tok:standing', 'pre:tok:walking', 'hyp:tok:alone', 'hyp:tok:swimming', 'hyp:tok:wearing', 'pre:tok:blue', 'hyp:tok:holding', 'pre:tok:cowboy', 'hyp:tag:prp$', 'hyp:tok:for', 'hyp:tok:desert', 'pre:tok:girl', 'pre:tok:on', 'hyp:tok:football', 'hyp:tok:home', 'oth:overlap:overlap75', 'pre:tok:blue', 'hyp:tok:outside', 'hyp:tok:outdoors', 'pre:tok:stage', 'hyp:tok:in', 'pre:tok:dog', 'hyp:tok:football', 'pre:tok:riding', 'hyp:tag:nns', 'hyp:tag:ex', 'hyp:tok:human', 'hyp:tok:there', 'oth:overlap:overlap75', 'pre:tok:blue', 'pre:tok:green', 'hyp:tag:prp$', 'hyp:tok:boy', 'hyp:tok:park', 'hyp:tag:,', 'hyp:tok:at', 'oth:overlap:overlap50', 'pre:tok:blue', 'hyp:tok:outside', 'pre:tok:blue', 'pre:tok:white', 'hyp:tok:outside', 'pre:tok:cooking', 'hyp:tag:in', 'hyp:tok:sleeping', 'hyp:tok:cat', 'hyp:tok:nobody', 'hyp:tok:outside', 'pre:tok:black', 'pre:tok:blue', 'hyp:tok:in', 'hyp:tok:cat', 'hyp:tok:girls', 'hyp:tok:sleeping', 'pre:tag:jj', 'hyp:tok:wearing', 'hyp:tok:there', 'pre:tok:beach', 'pre:tok:blue', 'hyp:tok:walking', 'hyp:tok:sleeping', 'pre:tok:dog', 'pre:tok:girl', 'pre:tok:girls', 'pre:tok:children', 'pre:tok:smiling', 'hyp:tag:prp$', 'hyp:tok:park', 'pre:tok:football', 'hyp:tok:pool', 'hyp:tok:concert', 'hyp:tok:in', 'pre:tok:sitting', 'hyp:tok:eating', 'oth:overlap:overlap75', 'pre:tok:blue', 'pre:tag:jj', 'hyp:tok:outdoors', 'hyp:tok:outside', 'pre:tag:nns', 'pre:tok:park', 'pre:tok:person', 'pre:tag:vbg', 'hyp:tag:ex', 'pre:tok:sleeping', 'hyp:tok:dinner', 'hyp:tok:in', 'hyp:tok:eating', 'hyp:tok:inside', 'pre:tag:in', 'hyp:tok:there', 'hyp:tok:wearing', 'hyp:tag:in', 'hyp:tok:eating', 'hyp:tok:driving', 'hyp:tok:his', 'hyp:tok:to', 'hyp:tok:tall', 'hyp:tok:old', 'hyp:tok:their', 'hyp:tag:.', 'hyp:tok:sleeping', 'hyp:tok:beach', 'pre:tok:in', 'pre:tok:water', 'pre:tok:cooking', 'pre:tok:top', 'hyp:tok:at', 'pre:tok:blue', 'hyp:tok:eating', 'hyp:tok:black', 'hyp:tag:prp$', 'hyp:tok:to', 'hyp:tag:nnp', 'pre:tok:dancing', 'hyp:tok:competition', 'hyp:tag:prp$', 'hyp:tok:boy', 'pre:tok:orange', 'hyp:tok:crowd', 'hyp:tok:dancing', 'pre:tag:dt', 'hyp:tok:running', 'hyp:tok:at', 'hyp:tok:in', 'hyp:tok:driving', 'hyp:tok:couch', 'hyp:tag:prp$', 'hyp:tok:at', 'hyp:tok:for', 'pre:tok:car', 'hyp:tok:tennis', 'hyp:tok:home', 'pre:tok:blue', 'pre:tok:black', 'pre:tok:drinking', 'pre:tok:on', 'pre:tok:train', 'pre:tok:race', 'hyp:tok:and', 'hyp:tag:.', 'pre:tok:boys', 'pre:tok:boy', 'pre:tok:standing', 'pre:tok:walking', 'hyp:tok:eating', 'hyp:tok:alone', 'pre:tok:blue', 'pre:tok:stage', 'pre:tok:singing', 'hyp:tok:eating', 'pre:tok:circle', 'hyp:tag:prp$', 'hyp:tok:walking', 'pre:tok:gathered', 'pre:tok:wall', 'hyp:tok:bar', 'hyp:tag:in', 'hyp:tok:nobody', 'hyp:tok:sitting', 'pre:tag:jj', 'pre:tok:standing', 'hyp:tok:there', 'pre:tag:vbp', 'pre:tok:guitar', 'pre:tok:skateboard', 'pre:tok:football', 'pre:tag:vbg', 'pre:tok:children', 'hyp:tok:there', 'hyp:tag:prp$', 'hyp:tok:to', 'hyp:tok:party', 'hyp:tok:baseball', 'pre:tag:nn', 'pre:tok:baseball', 'pre:tok:in', 'hyp:tok:eating', 'hyp:tag:prp$', 'oth:overlap:overlap75', 'hyp:tag:nnp', 'hyp:tag:in', 'pre:tok:beach', 'pre:tok:grass', 'hyp:tok:water', 'pre:tag:vbp', 'hyp:tok:tall', 'pre:tok:pool', 'pre:tag:dt', 'pre:tok:table', 'pre:tag:nn', 'hyp:tok:black', 'hyp:tok:in', 'pre:tok:girls', 'hyp:tok:cat', 'hyp:tag:in', 'pre:tok:dog', 'hyp:tok:swimming', 'hyp:tok:eating', 'hyp:tok:in', 'hyp:tok:sleeping', 'hyp:tok:eating', 'hyp:tok:inside', 'hyp:tok:for', 'hyp:tok:to', 'pre:tok:green', 'pre:tok:hands', 'hyp:tag:vb', 'hyp:tok:for', 'hyp:tok:their', 'hyp:tok:birthday', 'hyp:tok:having', 'hyp:tag:prp$', 'oth:overlap:overlap75', 'hyp:tag:in', 'pre:tok:sand', 'hyp:tok:woman', 'pre:tok:green', 'pre:tok:standing', 'pre:tok:walking', 'pre:tok:running', 'pre:tok:standing', 'pre:tok:field', 'pre:tok:skateboard', 'hyp:tag:prp$', 'hyp:tok:for', 'hyp:tok:park', 'hyp:tok:outside', 'hyp:tok:race', 'hyp:tag:in', 'hyp:tok:eating', 'hyp:tok:swimming', 'hyp:tok:eating', 'hyp:tok:in', 'hyp:tok:sleeping', 'hyp:tok:home', 'hyp:tag:in', 'hyp:tok:dancing', 'pre:tok:desk', 'hyp:tok:birthday', 'hyp:tag:in', 'pre:tok:sitting', 'hyp:tok:swimming', 'hyp:tok:for', 'hyp:tok:to', 'pre:tok:standing', 'hyp:tok:front', 'hyp:tag:.', 'pre:tok:walking', 'pre:tok:walks', 'pre:tok:skateboarder', 'pre:tok:blue', 'pre:tok:young', 'hyp:tok:sleeping', 'hyp:tok:cat', 'pre:tok:black', 'pre:tok:blue', 'pre:tok:pink', 'hyp:tok:girls', 'hyp:tok:nobody', 'oth:overlap:overlap75', 'pre:tok:blue', 'pre:tok:baseball', 'pre:tok:jumping', 'hyp:tag:prp$', 'hyp:tok:to', 'hyp:tok:for', 'pre:tok:riding', 'pre:tok:dancing', 'hyp:tag:prp$', 'hyp:tok:for', 'hyp:tok:at', 'hyp:tok:tall', 'pre:tag:dt', 'pre:tok:park', 'pre:tok:women', 'hyp:tag:prp$', 'pre:tok:boy', 'hyp:tok:at', 'hyp:tok:party', 'pre:tok:in', 'pre:tok:running', 'hyp:tok:sitting', 'pre:tok:black', 'pre:tok:brown', 'hyp:tok:park', 'hyp:tok:soccer', 'pre:tok:wooden', 'hyp:tag:jj', 'pre:tok:children', 'hyp:tok:running', 'hyp:tok:in', 'hyp:tok:inside', 'hyp:tok:cat', 'hyp:tok:sleeping', 'pre:tag:dt', 'hyp:tok:sleeping', 'hyp:tok:to', 'pre:tok:sitting', 'hyp:tok:their', 'pre:tok:sits', 'hyp:tok:outside', 'pre:tok:blue', 'hyp:tag:ex', 'hyp:tok:outdoors', 'pre:tok:room', 'hyp:tag:prp$', 'pre:tok:performing', 'hyp:tok:party', 'pre:tok:room', 'hyp:tok:his', 'pre:tok:children', 'hyp:tok:boy', 'hyp:tag:nnp', 'pre:tok:black', 'pre:tok:blue', 'hyp:tok:for', 'pre:tok:boy', 'hyp:tok:outside', 'pre:tok:blue', 'hyp:tok:train', 'pre:tok:blond', 'pre:tok:green', 'hyp:tok:in', 'pre:tok:sitting', 'pre:tok:holding', 'pre:tag:jj', 'hyp:tok:white', 'pre:tag:vbg', 'pre:tok:sitting', 'pre:tok:sits', 'hyp:tok:outdoors', 'hyp:tag:prp$', 'pre:tok:pool', 'hyp:tok:park', 'hyp:tok:baby', 'hyp:tok:lunch', 'hyp:tok:in', 'pre:tok:boy', 'hyp:tok:sleeping', 'hyp:tag:in', 'hyp:tok:dancing', 'pre:tok:store', 'hyp:tok:in', 'pre:tok:black', 'pre:tok:horse', 'hyp:tag:prp$', 'pre:tok:white', 'pre:tag:jj', 'pre:tok:yellow', 'hyp:tag:nns', 'pre:tok:red', 'hyp:tok:for', 'pre:tag:nn', 'hyp:tok:there', 'hyp:tok:outside', 'pre:tok:baseball', 'pre:tok:children', 'pre:tok:people', 'hyp:tok:running', 'hyp:tok:street', 'pre:tok:kids', 'pre:tag:nns', 'pre:tok:pole', 'hyp:tok:kids', 'oth:overlap:overlap75', 'pre:tok:snow', 'pre:tok:book', 'hyp:tok:race', 'hyp:tok:at', 'pre:tok:black', 'pre:tok:children', 'pre:tok:cooking', 'oth:overlap:overlap75', 'pre:tok:standing', 'hyp:tok:outside', 'hyp:tok:wearing', 'hyp:tok:near', 'pre:tok:little', 'pre:tok:young', 'pre:tok:children', 'pre:tok:blue', 'pre:tok:sitting', 'pre:tok:smiling', 'hyp:tok:party', 'pre:tag:jj', 'hyp:tok:outside', 'hyp:tok:for', 'oth:overlap:overlap50', 'pre:tok:blue', 'hyp:tag:prp$', 'hyp:tag:vb', 'hyp:tok:for', 'pre:tok:mountain', 'pre:tok:rock', 'hyp:tok:baseball', 'hyp:tok:in', 'hyp:tok:home', 'pre:tok:on', 'pre:tok:young', 'hyp:tok:tall', 'pre:tag:nns', 'pre:tok:food', 'hyp:tok:wearing', 'hyp:tok:there', 'pre:tag:.', 'hyp:tag:in', 'hyp:tag:.', 'hyp:tok:park', 'pre:tok:swimming', 'hyp:tok:watching', 'pre:tok:men', 'hyp:tok:football', 'pre:tok:snow', 'pre:tok:children', 'hyp:tag:prp$', 'hyp:tok:tall', 'hyp:tok:for', 'hyp:tok:after', 'hyp:tok:in', 'hyp:tok:inside', 'hyp:tok:cat', 'hyp:tok:eating', 'hyp:tok:in', 'hyp:tok:inside', 'hyp:tok:sleeping', 'hyp:tok:nobody', 'hyp:tok:in', 'hyp:tok:watching', 'hyp:tok:horse', 'hyp:tok:sleeping', 'hyp:tag:prp$', 'hyp:tok:for', 'hyp:tok:park', 'pre:tok:headphones', 'hyp:tok:catch', 'hyp:tok:for', 'hyp:tok:to', 'hyp:tok:their', 'hyp:tok:soccer', 'hyp:tok:birthday', 'hyp:tok:kids', 'hyp:tok:outside', 'hyp:tok:standing', 'hyp:tok:park', 'pre:tok:paint', 'pre:tok:blue', 'pre:tok:brown', 'pre:tok:orange', 'pre:tok:washing', 'pre:tag:jj', 'pre:tok:and', 'hyp:tok:sleeping', 'hyp:tok:swimming', 'hyp:tok:bike', 'hyp:tok:dog', 'pre:tok:standing', 'hyp:tok:park', 'hyp:tag:in', 'hyp:tok:running', 'hyp:tok:eating', 'pre:tok:blue', 'pre:tok:white', 'pre:tok:standing', 'hyp:tok:sleeping', 'pre:tok:blue', 'pre:tok:field', 'hyp:tok:car', 'pre:tok:snow', 'pre:tok:horse', 'hyp:tag:in', 'hyp:tok:outside', 'pre:tok:dancing', 'hyp:tok:beach', 'pre:tok:young', 'hyp:tok:park', 'hyp:tok:game', 'pre:tok:water', 'pre:tag:jj', 'hyp:tok:for', 'hyp:tok:to', 'hyp:tok:their', 'hyp:tok:bike', 'hyp:tok:snowboarder']
cofi_bert_25_sample10=sorted(cofi_bert_25_sample10)
cofi_bert_0_sample10=sorted(cofi_bert_0_sample10)
print(len(set(cofi_bert_0_sample10) & set(cofi_bert_25_sample10) )/len(set(cofi_bert_25_sample10)))

#in cofi 69% of concepts hollistically are the same tho the neuron behavior is different nad highest contrib is different


0.7804878048780488
0.6962025316455697


In [8]:
lth_bert_concepts25_sample3=['pre:tok:blue', 'pre:tok:large', 'pre:tok:green', 'hyp:tok:owner', 'oth:overlap:overlap75', 'pre:tok:blue', 'pre:tok:into', 'oth:overlap:overlap50', 'pre:tok:blue', 'hyp:tag:prp$', 'hyp:tok:for', 'hyp:tok:dogs', 'pre:tag:cd', 'hyp:tok:race', 'hyp:tok:tall', 'pre:tok:soccer', 'hyp:tag:prp$', 'oth:overlap:overlap50', 'hyp:tag:in', 'pre:tok:white', 'pre:tok:with', 'hyp:tok:outside', 'hyp:tok:tall', 'hyp:tok:in', 'hyp:tok:sleeping', 'hyp:tok:nobody', 'pre:tok:black', 'pre:tok:blue', 'pre:tok:jumping', 'pre:tok:cellphone', 'hyp:tag:prp$', 'pre:tok:green', 'pre:tok:guitar', 'pre:tok:sunglasses', 'oth:overlap:overlap50', 'pre:tok:blue', 'pre:tag:jj', 'pre:tok:laundry', 'oth:overlap:overlap50', 'pre:tok:blue', 'hyp:tok:football', 'pre:tag:vbp', 'pre:tok:pink', 'pre:tok:park', 'hyp:tok:in', 'hyp:tok:eating', 'hyp:tok:sleeping', 'hyp:tok:watching', 'pre:tok:blue', 'pre:tok:orange', 'pre:tok:white', 'hyp:tok:competition', 'pre:tok:children', 'pre:tok:young', 'pre:tok:boy', 'hyp:tok:alone', 'pre:tok:in', 'hyp:tok:sleeping', 'hyp:tok:sleeping', 'pre:tok:standing', 'hyp:tok:eating', 'hyp:tok:asleep', 'oth:overlap:overlap75', 'pre:tok:blue', 'hyp:tok:women', 'pre:tok:microphone', 'hyp:tok:song', 'pre:tok:blue', 'pre:tok:large', 'hyp:tok:friends', 'hyp:tok:couple', 'pre:tok:sunglasses', 'hyp:tok:his', 'hyp:tok:tall', 'hyp:tag:nnp', 'hyp:tok:their', 'hyp:tok:homeless', 'hyp:tok:is', 'hyp:tag:prp$', 'hyp:tok:outside', 'pre:tok:blue', 'hyp:tok:outdoors', 'hyp:tok:smiling', 'hyp:tok:something', 'hyp:tag:in', 'hyp:tok:swimming', 'pre:tok:snow', 'hyp:tok:in', 'pre:tok:running', 'hyp:tok:sleeping', 'hyp:tok:football', 'pre:tok:blue', 'pre:tok:girl', 'pre:tok:pool', 'hyp:tok:tall', 'pre:tag:dt', 'hyp:tok:ocean', 'hyp:tok:for', 'hyp:tok:to', 'hyp:tok:their', 'hyp:tok:in', 'hyp:tok:eating', 'hyp:tok:driving', 'hyp:tok:inside', 'hyp:tok:for', 'hyp:tok:to', 'hyp:tok:their', 'hyp:tag:prp$', 'pre:tag:vb', 'pre:tok:for', 'pre:tok:workers', 'hyp:tag:in', 'pre:tok:snow', 'hyp:tok:swimming', 'hyp:tag:cd', 'hyp:tok:man', 'pre:tok:yellow', 'pre:tag:jj', 'hyp:tag:ex', 'hyp:tok:person', 'pre:tok:young', 'pre:tok:children', 'hyp:tok:child', 'pre:tag:nn', 'hyp:tok:there', 'hyp:tok:wearing', 'hyp:tok:outdoors', 'hyp:tok:outside', 'hyp:tok:inside', 'hyp:tok:indoors', 'hyp:tok:kids', 'hyp:tok:in', 'hyp:tok:sitting', 'hyp:tok:sleeping', 'hyp:tok:nobody', 'hyp:tag:prp$', 'hyp:tok:playing', 'pre:tok:headphones', 'pre:tok:conversation', 'hyp:tag:nnp', 'hyp:tag:prp$', 'hyp:tag:prp', 'hyp:tok:park', 'hyp:tag:prp$', 'pre:tok:crowd', 'pre:tok:girls', 'hyp:tag:prp$', 'oth:overlap:overlap75', 'hyp:tok:for', 'pre:tok:boys', 'hyp:tok:boy', 'hyp:tag:prp$', 'hyp:tok:sleeping', 'hyp:tok:tall', 'hyp:tok:game', 'hyp:tag:in', 'pre:tok:riding', 'pre:tok:rock', 'pre:tok:children', 'pre:tok:girl', 'pre:tok:child', 'hyp:tok:eating', 'pre:tok:and', 'pre:tok:dog', 'pre:tok:white', 'hyp:tok:playing', 'hyp:tag:prp$', 'pre:tok:black', 'hyp:tok:for', 'hyp:tok:party', 'hyp:tok:sad', 'hyp:tag:prp$', 'hyp:tok:for', 'hyp:tok:about', 'hyp:tok:park', 'hyp:tok:after', 'hyp:tok:boy', 'oth:overlap:overlap75', 'pre:tok:pool', 'hyp:tok:kids', 'pre:tok:street', 'pre:tok:young', 'pre:tok:boy', 'pre:tok:playing', 'pre:tok:waiting', 'hyp:tok:at', 'hyp:tok:to', 'hyp:tok:their', 'hyp:tok:red', 'hyp:tok:lunch', 'hyp:tok:his', 'hyp:tok:walking', 'hyp:tok:their', 'pre:tok:green', 'hyp:tok:walk', 'pre:tok:blue', 'pre:tok:young', 'hyp:tok:nobody', 'hyp:tok:children', 'hyp:tok:at', 'hyp:tok:in', 'pre:tok:pool', 'hyp:tag:vb', 'hyp:tok:tall', 'hyp:tok:their', 'hyp:tok:friends', 'hyp:tok:funny', 'hyp:tok:there', 'oth:overlap:overlap50', 'pre:tok:blue', 'oth:overlap:overlap50', 'pre:tok:blue', 'hyp:tok:outside', 'pre:tag:jj', 'pre:tok:smiling', 'hyp:tag:prp$', 'hyp:tok:for', 'hyp:tok:to', 'hyp:tok:park', 'hyp:tok:race', 'hyp:tag:prp$', 'pre:tok:running', 'pre:tok:field', 'pre:tok:dog', 'oth:overlap:overlap75', 'pre:tok:young', 'pre:tok:table', 'hyp:tok:walking', 'pre:tag:vbp', 'pre:tok:playing', 'hyp:tok:black', 'hyp:tag:nnp', 'hyp:tag:prp$', 'hyp:tok:running', 'hyp:tok:family', 'oth:overlap:overlap50', 'pre:tok:blue', 'pre:tok:team', 'pre:tag:cc', 'pre:tok:woman', 'pre:tag:jj', 'hyp:tok:football', 'pre:tag:nns', 'pre:tag:nnp', 'hyp:tag:ex', 'hyp:tag:prp$', 'hyp:tok:playing', 'hyp:tok:for', 'hyp:tok:performing', 'hyp:tag:prp$', 'hyp:tok:to', 'hyp:tok:sad', 'hyp:tok:competition', 'hyp:tag:.', 'pre:tok:standing', 'pre:tok:street', 'hyp:tok:for', 'hyp:tok:cooking', 'hyp:tag:vb', 'hyp:tok:for', 'hyp:tok:their', 'hyp:tok:he', 'pre:tok:young', 'pre:tok:children', 'pre:tok:kids', 'hyp:tok:in', 'hyp:tok:inside', 'hyp:tok:eating', 'hyp:tok:driving', 'pre:tok:water', 'pre:tok:young', 'pre:tok:pool', 'pre:tok:children', 'hyp:tag:prp$', 'pre:tok:soccer', 'hyp:tok:cat', 'hyp:tok:dinner', 'pre:tag:jj', 'hyp:tag:vb', 'hyp:tok:for', 'hyp:tag:prp$', 'hyp:tok:food', 'hyp:tok:snowboarder', 'pre:tok:red', 'pre:tok:young', 'pre:tok:wedding', 'hyp:tag:nnp', 'hyp:tag:prp$', 'hyp:tok:park', 'hyp:tok:about', 'hyp:tok:sad', 'hyp:tag:prp$', 'hyp:tok:to', 'pre:tag:jj', 'hyp:tok:black', 'pre:tok:sunglasses', 'pre:tok:sits', 'pre:tok:sitting', 'pre:tok:children', 'hyp:tag:,', 'pre:tok:child', 'hyp:tok:in', 'oth:overlap:overlap75', 'hyp:tok:at', 'pre:tag:dt', 'pre:tag:vb', 'hyp:tok:outside', 'hyp:tok:to', 'pre:tok:blue', 'hyp:tok:for', 'hyp:tok:tall', 'hyp:tok:for', 'hyp:tok:to', 'hyp:tag:nnp', 'hyp:tok:their', 'hyp:tag:prp$', 'hyp:tok:to', 'hyp:tok:children', 'hyp:tok:playing', 'hyp:tok:kids', 'pre:tok:jumps', 'hyp:tok:inside', 'hyp:tok:her', 'hyp:tok:their', 'hyp:tok:after', 'hyp:tok:rain', 'hyp:tok:fishing', 'hyp:tag:prp', 'hyp:tag:vb', 'hyp:tok:their', 'hyp:tok:tall', 'hyp:tok:for', 'oth:overlap:overlap75', 'pre:tok:blue', 'pre:tag:vb', 'pre:tag:jj', 'pre:tok:car', 'hyp:tok:sitting', 'pre:tok:blue', 'hyp:tok:nobody', 'hyp:tok:cat', 'hyp:tok:in', 'hyp:tok:nobody', 'hyp:tok:beach', 'hyp:tok:dog', 'pre:tok:blue', 'hyp:tok:nobody', 'hyp:tag:prp$', 'pre:tok:riding', 'pre:tok:through', 'hyp:tok:for', 'hyp:tok:friends', 'hyp:tag:in', 'hyp:tok:outside', 'hyp:tag:prp$', 'hyp:tag:prp$', 'hyp:tok:at', 'hyp:tok:for', 'hyp:tok:football', 'pre:tok:air', 'pre:tok:park', 'pre:tok:sitting', 'pre:tok:children', 'pre:tok:baby', 'pre:tag:rb', 'pre:tok:blue', 'hyp:tok:standing', 'hyp:tok:inside', 'pre:tok:blue', 'hyp:tok:eating', 'hyp:tag:nnp', 'hyp:tag:prp$', 'hyp:tok:tall', 'hyp:tok:about']
lth_bert_concepts0_sample3=['hyp:tag:prp$', 'hyp:tok:for', 'hyp:tag:vb', 'hyp:tok:playing', 'pre:tok:air', 'hyp:tok:sleeping', 'pre:tok:standing', 'hyp:tok:men', 'pre:tok:waiting', 'hyp:tok:lake', 'hyp:tok:for', 'pre:tok:blue', 'hyp:tok:tall', 'hyp:tok:to', 'oth:overlap:overlap75', 'pre:tok:blue', 'hyp:tok:park', 'hyp:tok:playing', 'hyp:tok:in', 'hyp:tok:eating', 'hyp:tok:inside', 'hyp:tok:watching', 'hyp:tok:for', 'hyp:tok:her', 'hyp:tok:tall', 'hyp:tok:sad', 'hyp:tok:their', 'hyp:tag:prp$', 'hyp:tok:to', 'hyp:tok:for', 'hyp:tok:woman', 'pre:tok:shirt', 'pre:tok:blue', 'hyp:tok:there', 'oth:overlap:overlap50', 'pre:tok:blue', 'hyp:tok:standing', 'oth:overlap:overlap50', 'pre:tok:blue', 'hyp:tok:playing', 'pre:tag:.', 'hyp:tok:watching', 'hyp:tok:outside', 'pre:tok:blue', 'hyp:tok:outdoors', 'hyp:tok:vacation', 'pre:tok:black', 'pre:tok:blue', 'pre:tag:nnp', 'hyp:tok:outside', 'hyp:tag:in', 'pre:tok:with', 'hyp:tok:walking', 'hyp:tok:swimming', 'oth:overlap:overlap50', 'pre:tok:blue', 'hyp:tok:women', 'hyp:tok:in', 'hyp:tok:sleeping', 'hyp:tok:cat', 'hyp:tok:inside', 'hyp:tag:prp$', 'hyp:tok:tall', 'hyp:tok:old', 'hyp:tok:they', 'hyp:tok:sad', 'pre:tag:nns', 'pre:tok:standing', 'hyp:tag:in', 'pre:tok:pool', 'hyp:tok:eating', 'pre:tag:jj', 'hyp:tok:outside', 'hyp:tag:in', 'hyp:tok:eating', 'hyp:tok:nobody', 'hyp:tok:running', 'hyp:tok:woman', 'pre:tok:walking', 'pre:tok:standing', 'hyp:tag:.', 'hyp:tok:red', 'hyp:tok:eating', 'hyp:tok:in', 'pre:tok:field', 'pre:tok:grass', 'pre:tok:blue', 'pre:tok:man', 'hyp:tok:to', 'hyp:tok:for', 'hyp:tok:to', 'hyp:tok:tall', 'hyp:tok:their', 'pre:tag:cd', 'pre:tok:red', 'hyp:tok:sleeping', 'hyp:tok:there', 'pre:tok:girl', 'pre:tok:woman', 'hyp:tok:her', 'pre:tok:wooden', 'hyp:tok:in', 'hyp:tok:sleeping', 'hyp:tok:swimming', 'hyp:tok:naked', 'hyp:tok:for', 'hyp:tok:to', 'hyp:tok:their', 'pre:tag:jj', 'pre:tok:sitting', 'hyp:tag:.', 'hyp:tag:in', 'hyp:tok:sleeping', 'hyp:tok:eating', 'hyp:tag:.', 'hyp:tok:concert', 'hyp:tok:outdoors', 'hyp:tok:outside', 'pre:tok:race', 'pre:tok:pool', 'hyp:tok:kids', 'pre:tag:cc', 'pre:tok:pool', 'pre:tok:beach', 'pre:tok:smiling', 'pre:tag:jj', 'hyp:tok:outside', 'hyp:tok:walking', 'hyp:tok:inside', 'pre:tok:blue', 'hyp:tok:eating', 'pre:tok:blue', 'pre:tok:smiling', 'pre:tok:blond', 'hyp:tok:soccer', 'hyp:tok:school', 'hyp:tag:prp$', 'pre:tok:sitting', 'hyp:tok:for', 'hyp:tok:tall', 'hyp:tag:nns', 'hyp:tok:there', 'pre:tok:pink', 'hyp:tok:human', 'pre:tok:standing', 'pre:tok:stands', 'pre:tok:wall', 'pre:tok:older', 'hyp:tok:sleeping', 'pre:tok:blue', 'hyp:tok:sitting', 'hyp:tok:men', 'hyp:tag:prp$', 'hyp:tok:men', 'pre:tok:pool', 'pre:tok:skateboard', 'pre:tok:and', 'hyp:tok:swimming', 'hyp:tok:dinner', 'hyp:tok:naked', 'hyp:tag:in', 'pre:tok:construction', 'hyp:tag:in', 'pre:tok:elderly', 'pre:tok:talking', 'pre:tok:smiling', 'hyp:tok:walking', 'pre:tok:blue', 'pre:tok:train', 'hyp:tok:walks', 'hyp:tok:stand', 'hyp:tag:nnp', 'hyp:tag:prp$', 'pre:tok:microphone', 'pre:tok:talking', 'hyp:tok:outside', 'pre:tok:blue', 'hyp:tok:outdoors', 'pre:tok:ball', 'oth:overlap:overlap50', 'pre:tag:jj', 'pre:tok:blue', 'pre:tok:smiling', 'pre:tok:smiles', 'hyp:tag:in', 'pre:tok:field', 'hyp:tag:.', 'pre:tok:running', 'hyp:tok:playing', 'pre:tok:black', 'pre:tok:children', 'pre:tok:pink', 'hyp:tok:in', 'hyp:tok:inside', 'hyp:tok:watching', 'hyp:tok:swimming', 'hyp:tok:for', 'hyp:tok:to', 'pre:tok:standing', 'hyp:tok:for', 'hyp:tok:to', 'hyp:tok:her', 'hyp:tok:their', 'hyp:tok:after', 'pre:tag:jj', 'hyp:tok:to', 'hyp:tag:prp$', 'hyp:tok:man', 'pre:tag:jj', 'hyp:tok:boy', 'hyp:tok:party', 'hyp:tok:for', 'pre:tok:black', 'pre:tok:blue', 'pre:tok:skateboard', 'pre:tok:children', 'pre:tok:people', 'pre:tok:eating', 'hyp:tag:in', 'pre:tok:water', 'pre:tok:pool', 'hyp:tok:eating', 'hyp:tag:prp$', 'hyp:tag:vb', 'hyp:tok:for', 'hyp:tok:doing', 'pre:tok:air', 'pre:tag:.', 'hyp:tok:for', 'hyp:tok:to', 'hyp:tok:their', 'hyp:tok:competition', 'hyp:tag:vb', 'hyp:tok:for', 'hyp:tok:tall', 'hyp:tag:prp$', 'hyp:tag:in', 'hyp:tok:sleeping', 'hyp:tok:watching', 'hyp:tok:for', 'pre:tok:playing', 'pre:tok:children', 'pre:tok:volleyball', 'hyp:tok:sitting', 'pre:tok:running', 'pre:tok:children', 'pre:tok:field', 'hyp:tag:vb', 'pre:tok:standing', 'hyp:tok:lady', 'pre:tok:trick', 'hyp:tok:eating', 'hyp:tok:in', 'hyp:tok:sleeping', 'hyp:tok:singing', 'pre:tag:.', 'hyp:tok:dog', 'hyp:tag:vb', 'hyp:tok:for', 'pre:tok:standing', 'hyp:tok:her', 'hyp:tok:their', 'hyp:tok:she', 'hyp:tok:sad', 'hyp:tok:house', 'pre:tag:jj', 'hyp:tok:running', 'hyp:tok:people', 'pre:tok:blue', 'hyp:tok:standing', 'hyp:tok:no', 'pre:tag:in', 'hyp:tag:prp$', 'hyp:tok:for', 'hyp:tok:to', 'pre:tok:tree', 'pre:tok:cooking', 'hyp:tag:prp$', 'hyp:tag:vb', 'hyp:tok:for', 'hyp:tok:sad', 'hyp:tok:about', 'hyp:tag:nnp', 'hyp:tag:prp$', 'hyp:tok:tall', 'hyp:tok:for']
lth_bert_concepts25_sample3=sorted(lth_bert_concepts25_sample3)
lth_bert_concepts0_sample3=sorted(lth_bert_concepts0_sample3)
#so formulas than encompass sentence 3 in the sparse and dense holistically cover very different concept
#original idea was to look at all the concepts and in theory they shoulve been difference if lth doesnt reconstruct neurons 
because i wanted to see how , to classify sample 3, it is learning a new path. so are the concepts more oritened towards something in sparse over dense. answer they are less 
by the cce results. this makes sense his highest neuron behavior is alos very different on avg for any given sample (comparing that sample at sparse and dense)
so basically even expls are saying that concepts it explains are very different and also cover different abstractions (44% of concepts covered in 25% are in the dense and. 62.5% of abstractinos covered in sample 3 are the ones in the dense)

takeaway to make the same classification both are learning vastly different concepts at the highest contrib neuron, holistcaly acorss all neurons, and neuron behavior is different


In [32]:
len(set(lth_bert_concepts0_sample3) & set(lth_bert_concepts25_sample3) )/len(set(lth_bert_concepts25_sample3))

0.4429530201342282

In [25]:
import pickle
from collections import Counter
with open("/workspace/CCE_NLI/code/Abstractions/final_abstractions.pkl", 'rb') as f:
    abs_map = pickle.load(f)

def find_cluster(raw_concept):
    for cluster in abs_map:
        if raw_concept in abs_map[cluster]:
            return cluster
def find_abstractions(expls):
    glob=[expls]
    abstracts=[]
    exact_concepts_perabs=defaultdict(set)
    for concept in glob:
        raw_concept = concept.split(":")[-1]
        abstraction_cluster = find_cluster(raw_concept)
        if abstraction_cluster==143: 
            continue
        if not abstraction_cluster:abstraction_cluster=150
        exact_concepts_perabs[abstraction_cluster].add(raw_concept)
        
        abstracts.append(abstraction_cluster)
    return Counter(abstracts), abstracts, exact_concepts_perabs
cluses=[]
for c in concepts0_sample3:
    _, clus, _ = find_abstractions(c)
    cluses.extend(clus)
Counter(cluses)

Counter({6: 24,
         66: 18,
         73: 16,
         74: 14,
         89: 13,
         59: 12,
         85: 11,
         79: 10,
         16: 10,
         87: 9,
         27: 9,
         118: 9,
         78: 7,
         112: 7,
         26: 5,
         135: 5,
         129: 5,
         60: 4,
         18: 4,
         45: 4,
         22: 4,
         62: 3,
         1: 3,
         150: 3,
         111: 3,
         142: 3,
         102: 2,
         17: 2,
         48: 2,
         100: 2,
         24: 2,
         52: 2,
         25: 2,
         54: 1,
         88: 1,
         115: 1,
         132: 1,
         9: 1,
         56: 1,
         12: 1,
         76: 1,
         58: 1,
         81: 1,
         75: 1,
         13: 1,
         105: 1,
         67: 1,
         42: 1,
         30: 1,
         95: 1,
         50: 1,
         128: 1,
         8: 1})

In [24]:
lth_bert_run25_sample3=Counter(cluses)

In [26]:
lth_bert_run0_sample3=Counter(cluses)

In [31]:
len(set(run25_sample3.keys()) & set(run0_sample3.keys())) / len(set(run25_sample3.keys()))

0.625